# Stage 1: Build the monthly modeling dataset
# Forecasting goal: predict next month's demand units for a specific product in a specific region.
# Demand definition: Quantity from all valid orders; delayed orders remain valid demand.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "Python" / "run_sql_analysis.py").is_file()
)
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "sales_orders.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "monthly_demand.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = {
    "Order_ID",
    "Order_Date",
    "Product_ID",
    "Region_ID",
    "Quantity",
}

# 1. Load raw orders and verify that all modeling fields are present.
sales = pd.read_csv(DATA_PATH)
missing_columns = REQUIRED_COLUMNS.difference(sales.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

# 2. Standardize types and expose invalid dates or quantities before modeling.
sales["Order_Date"] = pd.to_datetime(sales["Order_Date"], errors="coerce")
sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce")

quality_report = pd.Series(
    {
        "duplicate_order_ids": sales["Order_ID"].duplicated().sum(),
        "missing_required_values": sales[list(REQUIRED_COLUMNS)].isna().any(axis=1).sum(),
        "invalid_dates": sales["Order_Date"].isna().sum(),
        "non_positive_quantity": (sales["Quantity"] <= 0).sum(),
    },
    name="value",
)

if (quality_report > 0).any():
    raise ValueError(f"Data quality checks failed:\n{quality_report[quality_report > 0]}")

# 3. Aggregate to the forecasting grain: month x product x region.
sales["Month"] = sales["Order_Date"].dt.to_period("M").dt.to_timestamp()
monthly_observed = (
    sales.groupby(["Month", "Product_ID", "Region_ID"], as_index=False)["Quantity"]
    .sum()
    .rename(columns={"Quantity": "Actual_Demand"})
)

# 4. Complete the product-region-month panel; no orders means zero demand.
months = pd.date_range(
    monthly_observed["Month"].min(),
    monthly_observed["Month"].max(),
    freq="MS",
)
products = sales["Product_ID"].drop_duplicates().sort_values()
regions = sales["Region_ID"].drop_duplicates().sort_values()
full_index = pd.MultiIndex.from_product(
    [months, products, regions],
    names=["Month", "Product_ID", "Region_ID"],
)

modeling_data = (
    monthly_observed.set_index(["Month", "Product_ID", "Region_ID"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)
modeling_data["Actual_Demand"] = modeling_data["Actual_Demand"].astype("int64")
modeling_data["Year"] = modeling_data["Month"].dt.year
modeling_data["Month_Number"] = modeling_data["Month"].dt.month
modeling_data["Time_Index"] = (
    (modeling_data["Month"].dt.year - modeling_data["Month"].dt.year.min()) * 12
    + modeling_data["Month"].dt.month
)

# 5. Assert that every product-region series has a complete monthly timeline.
expected_rows = len(months) * products.nunique() * regions.nunique()
assert len(modeling_data) == expected_rows
assert modeling_data[["Month", "Product_ID", "Region_ID"]].duplicated().sum() == 0
assert modeling_data["Actual_Demand"].ge(0).all()
assert modeling_data["Month"].nunique() == len(months)

modeling_data.to_csv(OUTPUT_PATH, index=False)

print(f"Raw rows: {len(sales):,}")
print(f"Modeling rows: {len(modeling_data):,}")
print(f"Time span: {months.min():%Y-%m} to {months.max():%Y-%m}")
print(f"Products x regions: {products.nunique()} x {regions.nunique()}")
print(f"Saved: {OUTPUT_PATH}")
modeling_data.head(10)

Raw rows: 10,000
Modeling rows: 1,152
Time span: 2023-01 to 2025-12
Products x regions: 8 x 4
Saved: C:\Users\circl\Documents\New folder\Demand-Forecasting-Analytics\data\processed\monthly_demand.csv


,Month,Product_ID,Region_ID,Actual_Demand,Year,Month_Number,Time_Index
0,2023-01-01,P001,R01,7133,2023,1,1
1,2023-01-01,P001,R02,8693,2023,1,1
2,2023-01-01,P001,R03,7685,2023,1,1
3,2023-01-01,P001,R04,13359,2023,1,1
4,2023-01-01,P002,R01,8414,2023,1,1
5,2023-01-01,P002,R02,13088,2023,1,1
6,2023-01-01,P002,R03,4146,2023,1,1
7,2023-01-01,P002,R04,17017,2023,1,1
8,2023-01-01,P003,R01,7511,2023,1,1
9,2023-01-01,P003,R02,6622,2023,1,1


# Stage 2: Create historical features and development/final test split
# All features use data from before the current month; the target is current-month Actual_Demand.

In [2]:
FEATURE_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "demand_features.csv"


SERIES_KEYS = ["Product_ID", "Region_ID"]

LAG_WINDOWS = [1, 2, 3, 6, 12]
ROLLING_WINDOWS = [3, 6]


# ---------------------------------------------------------
# 1. Sort each Product x Region time series chronologically
# ---------------------------------------------------------

features = (
    modeling_data
    .sort_values(SERIES_KEYS + ["Month"])
    .copy()
)

series = features.groupby(
    SERIES_KEYS,
    sort=False
)["Actual_Demand"]


# ---------------------------------------------------------
# 2. Lag features
# ---------------------------------------------------------
# Example:
# Demand_Lag_1 for August = July demand
# Demand_Lag_2 for August = June demand

for lag in LAG_WINDOWS:
    features[f"Demand_Lag_{lag}"] = series.shift(lag)


# ---------------------------------------------------------
# 3. Rolling demand features
# ---------------------------------------------------------
# shift(1) is critical:
# the current month's Actual_Demand must NOT be included
# when predicting the current month.

for window in ROLLING_WINDOWS:
    features[f"Demand_Rolling_Mean_{window}"] = (
        series.transform(
            lambda values:
            values.shift(1)
            .rolling(
                window=window,
                min_periods=window
            )
            .mean()
        )
    )


# ---------------------------------------------------------
# 4. Calendar / seasonality features
# ---------------------------------------------------------

features["Month_Number"] = features["Month"].dt.month
features["Quarter"] = features["Month"].dt.quarter

features["Time_Index"] = (
    (features["Month"].dt.year - features["Month"].dt.year.min()) * 12
    + features["Month"].dt.month
)


# ---------------------------------------------------------
# 5. Define historical features that require warm-up
# ---------------------------------------------------------

historical_feature_columns = [
    *(f"Demand_Lag_{lag}" for lag in LAG_WINDOWS),
    *(f"Demand_Rolling_Mean_{window}" for window in ROLLING_WINDOWS),
]


# Rows in the first 12 months do not have Lag_12,
# so they cannot yet be used for model training.

usable_features = (
    features
    .dropna(subset=historical_feature_columns)
    .copy()
)


# ---------------------------------------------------------
# 6. Reserve latest 6 months as untouched final test set
# ---------------------------------------------------------

final_test_months = 6

final_test_start = (
    usable_features["Month"].max()
    - pd.DateOffset(months=final_test_months - 1)
)

development_data = (
    usable_features[
        usable_features["Month"] < final_test_start
    ]
    .copy()
)

final_test_data = (
    usable_features[
        usable_features["Month"] >= final_test_start
    ]
    .copy()
)


# ---------------------------------------------------------
# 7. QA checks and model-ready interfaces
# ---------------------------------------------------------

assert not development_data.empty
assert not final_test_data.empty

assert (
    development_data["Month"].max()
    <
    final_test_data["Month"].min()
)

assert final_test_data["Month"].nunique() == final_test_months

assert development_data[
    historical_feature_columns
].notna().all().all()

assert final_test_data[
    historical_feature_columns
].notna().all().all()

model_feature_columns = [
    *historical_feature_columns,
    "Month_Number",
    "Quarter",
    "Time_Index",
]

X_dev = development_data[model_feature_columns].copy()
y_dev = development_data["Actual_Demand"].copy()
X_test = final_test_data[model_feature_columns].copy()
y_test = final_test_data["Actual_Demand"].copy()


# ---------------------------------------------------------
# 8. Save feature dataset
# ---------------------------------------------------------

features.to_csv(
    FEATURE_OUTPUT_PATH,
    index=False
)


# ---------------------------------------------------------
# 9. Summary
# ---------------------------------------------------------

print(f"Feature rows: {len(features):,}")

print(
    f"Usable rows after lag warm-up: "
    f"{len(usable_features):,}"
)

print(
    f"Development rows: "
    f"{len(development_data):,}"
)

print(
    f"Final test rows: "
    f"{len(final_test_data):,}"
)

print(
    f"Development period: "
    f"{development_data['Month'].min():%Y-%m} "
    f"to "
    f"{development_data['Month'].max():%Y-%m}"
)

print(
    f"Final test period: "
    f"{final_test_data['Month'].min():%Y-%m} "
    f"to "
    f"{final_test_data['Month'].max():%Y-%m}"
)

print(
    f"Feature file: "
    f"{FEATURE_OUTPUT_PATH}"
)


final_test_data[
    [
        "Month",
        "Product_ID",
        "Region_ID",
        "Actual_Demand",
        *historical_feature_columns,
        "Month_Number",
        "Quarter",
        "Time_Index",
    ]
].head(10)

Feature rows: 1,152
Usable rows after lag warm-up: 768
Development rows: 576
Final test rows: 192
Development period: 2024-01 to 2025-06
Final test period: 2025-07 to 2025-12
Feature file: C:\Users\circl\Documents\New folder\Demand-Forecasting-Analytics\data\processed\demand_features.csv


,Month,Product_ID,Region_ID,Actual_Demand,Demand_Lag_1,Demand_Lag_2,Demand_Lag_3,Demand_Lag_6,Demand_Lag_12,Demand_Rolling_Mean_3,Demand_Rolling_Mean_6,Month_Number,Quarter,Time_Index
960,2025-07-01,P001,R01,3906,6170.0,8022.0,7581.0,5452.0,6193.0,7257.666667,6565.500000,7,3,31
992,2025-08-01,P001,R01,13590,3906.0,6170.0,8022.0,8554.0,11369.0,6032.666667,6307.833333,8,3,32
1024,2025-09-01,P001,R01,9236,13590.0,3906.0,6170.0,3614.0,5908.0,7888.666667,7147.166667,9,3,33
1056,2025-10-01,P001,R01,5834,9236.0,13590.0,3906.0,7581.0,11360.0,8910.666667,8084.166667,10,4,34
1088,2025-11-01,P001,R01,10025,5834.0,9236.0,13590.0,8022.0,18120.0,9553.333333,7793.000000,11,4,35
1120,2025-12-01,P001,R01,12233,10025.0,5834.0,9236.0,6170.0,12568.0,8365.000000,8126.833333,12,4,36
961,2025-07-01,P001,R02,16164,13317.0,12796.0,11323.0,11387.0,13842.0,12478.666667,11168.000000,7,3,31
993,2025-08-01,P001,R02,8341,16164.0,13317.0,12796.0,9822.0,11770.0,14092.333333,11964.166667,8,3,32
1025,2025-09-01,P001,R02,12766,8341.0,16164.0,13317.0,8363.0,11121.0,12607.333333,11717.333333,9,3,33
1057,2025-10-01,P001,R02,11017,12766.0,8341.0,16164.0,11323.0,12421.0,12423.666667,12451.166667,10,4,34


In [3]:
from sklearn.model_selection import TimeSeriesSplit


# ---------------------------------------------------------
# Stage 3: Build development-only monthly CV splits
# ---------------------------------------------------------
# Split unique months first, then map each fold back to panel rows.
# This keeps every product-region row from the same month together.

CV_SPLITS = 3
development_months = pd.Index(
    development_data["Month"]
    .drop_duplicates()
    .sort_values()
)

if len(development_months) <= CV_SPLITS:
    raise ValueError(
        "Development data must contain more months than CV_SPLITS."
    )

month_splitter = TimeSeriesSplit(n_splits=CV_SPLITS)
development_cv_splits = []

for fold_number, (train_month_positions, validation_month_positions) in enumerate(
    month_splitter.split(development_months),
    start=1,
):
    train_months = development_months[train_month_positions]
    validation_months = development_months[validation_month_positions]

    train_row_labels = development_data.index[
        development_data["Month"].isin(train_months)
    ]
    validation_row_labels = development_data.index[
        development_data["Month"].isin(validation_months)
    ]

    train_rows = X_dev.index.get_indexer(train_row_labels)
    validation_rows = X_dev.index.get_indexer(validation_row_labels)

    assert train_rows.min() >= 0
    assert validation_rows.min() >= 0
    assert train_months.max() < validation_months.min()

    development_cv_splits.append((train_rows, validation_rows))

    print(
        f"CV fold {fold_number}: "
        f"{train_months.min():%Y-%m} to {train_months.max():%Y-%m} "
        f"-> "
        f"{validation_months.min():%Y-%m} to {validation_months.max():%Y-%m}"
    )

print(f"Development CV folds: {len(development_cv_splits)}")
print("Final test data is excluded from development CV.")

CV fold 1: 2024-01 to 2024-06 -> 2024-07 to 2024-10
CV fold 2: 2024-01 to 2024-10 -> 2024-11 to 2025-02
CV fold 3: 2024-01 to 2025-02 -> 2025-03 to 2025-06
Development CV folds: 3
Final test data is excluded from development CV.


In [4]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline


# ---------------------------------------------------------
# Stage 4: Compare and tune models on development data only
# ---------------------------------------------------------

model_specs = {
    "mean_baseline": (
        Pipeline([
            ("model", DummyRegressor()),
        ]),
        {
            "model__strategy": ["mean"],
        },
    ),
    "random_forest": (
        Pipeline([
            ("model", RandomForestRegressor(
                random_state=42,
                n_jobs=-1,
            )),
        ]),
        {
            "model__n_estimators": [200],
            "model__max_depth": [None, 12],
            "model__min_samples_leaf": [1, 3],
        },
    ),
    "hist_gradient_boosting": (
        Pipeline([
            ("model", HistGradientBoostingRegressor(
                random_state=42,
            )),
        ]),
        {
            "model__learning_rate": [0.05, 0.1],
            "model__max_iter": [100, 200],
            "model__max_leaf_nodes": [15, 31],
        },
    ),
}

cv_results = []
searches = {}

for model_name, (pipeline, parameter_grid) in model_specs.items():
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        scoring="neg_mean_absolute_error",
        cv=development_cv_splits,
        refit=True,
        n_jobs=-1,
        error_score="raise",
    )
    search.fit(X_dev, y_dev)

    searches[model_name] = search
    cv_results.append(
        {
            "Model": model_name,
            "CV_MAE": -search.best_score_,
            "Best_Params": search.best_params_,
        }
    )

cv_results = (
    pd.DataFrame(cv_results)
    .sort_values("CV_MAE")
    .reset_index(drop=True)
)

best_model_name = cv_results.loc[0, "Model"]
best_search = searches[best_model_name]
best_model = best_search.best_estimator_

# The selected model was refit on all development rows by GridSearchCV.
test_predictions = best_model.predict(X_test)
final_test_metrics = pd.Series(
    {
        "MAE": mean_absolute_error(y_test, test_predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),
    },
    name="Final test metric",
)

print("Development CV results:")
print(cv_results[["Model", "CV_MAE"]])
print(f"Selected model: {best_model_name}")
print(f"Best parameters: {best_search.best_params_}")
print("Final test metrics:")
print(final_test_metrics)

cv_results

Development CV results:
                    Model       CV_MAE
0           random_forest  2997.849257
1  hist_gradient_boosting  3027.321994
2           mean_baseline  3777.366193
Selected model: random_forest
Best parameters: {'model__max_depth': 12, 'model__min_samples_leaf': 3, 'model__n_estimators': 200}
Final test metrics:
MAE     3141.396322
RMSE    4245.651768
Name: Final test metric, dtype: float64


,Model,CV_MAE,Best_Params
0,random_forest,2997.849257,"{'model__max_depth': 12, 'model__min_samples_l..."
1,hist_gradient_boosting,3027.321994,"{'model__learning_rate': 0.05, 'model__max_ite..."
2,mean_baseline,3777.366193,{'model__strategy': 'mean'}


# Stage 5: Final Model Evaluation & Benchmarking

Compare all three forecasts on the same final test rows (July-December 2025):

- **Naive Forecast**: use the previous month's actual demand (`Demand_Lag_1`).
- **Seasonal Naive**: use the same month's actual demand from the previous year (`Demand_Lag_12`).
- **Random Forest**: use the model tuned and refit on development data in Stage 4.

MAE and RMSE are measured in demand units; lower is better. This is a rolling
one-month-ahead evaluation: each month's historical features include actuals
available before that month, including earlier test months. It is not a single
six-month-ahead forecast. The Random Forest stays fixed during the test period;
these results are for reporting, not further tuning on the final test set.


In [5]:
# Keep actuals and every forecast aligned to exactly the same test rows.
assert X_test.index.equals(y_test.index)
assert X_test.index.equals(final_test_data.index)
assert y_test.equals(final_test_data["Actual_Demand"])

benchmark_predictions = pd.DataFrame(index=y_test.index)
benchmark_predictions["Naive Forecast"] = X_test["Demand_Lag_1"]
benchmark_predictions["Seasonal Naive"] = X_test["Demand_Lag_12"]

# Explicitly retrieve Random Forest, even if another model wins a future CV run.
# best_estimator_ has already been refit on development data only.
random_forest_model = searches["random_forest"].best_estimator_
benchmark_predictions["Random Forest"] = random_forest_model.predict(X_test)

assert np.isfinite(y_test.to_numpy()).all()
assert np.isfinite(benchmark_predictions.to_numpy()).all()

benchmark_results = pd.DataFrame([
    {
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
    }
    for model_name, predictions in benchmark_predictions.items()
]).sort_values("MAE").reset_index(drop=True)

print(
    f"Final test: {final_test_data['Month'].min():%Y-%m} "
    f"to {final_test_data['Month'].max():%Y-%m} | "
    f"{len(y_test):,} observations | lower is better"
)

# Round for display only; retain full precision in benchmark_results.
benchmark_results.round(2)


Final test: 2025-07 to 2025-12 | 192 observations | lower is better


,Model,MAE,RMSE
0,Random Forest,3141.40,4245.65
1,Seasonal Naive,3899.38,4910.20
2,Naive Forecast,4258.60,5450.38


# Stage 6: Export model forecast results

Export the final-test predictions to `outputs/forecasts/model_forecast_results.csv`, one row
per month, product and region. These are historical test predictions, not future
forecasts. `Forecast_Demand` is the Random Forest prediction.
`Forecast_Error = Forecast_Demand - Actual_Demand`: positive means overforecast,
negative means underforecast. Keep full numeric precision for downstream metrics.


In [6]:
FORECAST_OUTPUT_PATH = PROJECT_ROOT / "outputs" / "forecasts" / "model_forecast_results.csv"
FORECAST_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

assert benchmark_predictions.index.equals(final_test_data.index)
model_forecast_results = final_test_data[
    ["Month", "Product_ID", "Region_ID", "Actual_Demand"]
].join(benchmark_predictions.rename(columns={
    "Random Forest": "Forecast_Demand",
    "Naive Forecast": "Naive_Forecast",
    "Seasonal Naive": "Seasonal_Naive_Forecast",
}))
model_forecast_results["Forecast_Error"] = (
    model_forecast_results["Forecast_Demand"]
    - model_forecast_results["Actual_Demand"]
)
model_forecast_results["Absolute_Error"] = (
    model_forecast_results["Forecast_Error"].abs()
)
model_forecast_results["Squared_Error"] = (
    model_forecast_results["Forecast_Error"] ** 2
)
model_forecast_results = model_forecast_results.sort_values(
    ["Month", "Product_ID", "Region_ID"]
).reset_index(drop=True)

assert len(model_forecast_results) == len(y_test)
assert not model_forecast_results.duplicated(
    ["Month", "Product_ID", "Region_ID"]
).any()
assert model_forecast_results.notna().all().all()

model_forecast_results.to_csv(
    FORECAST_OUTPUT_PATH, index=False, date_format="%Y-%m-%d", encoding="utf-8-sig"
)

# Validate the saved CSV and reconcile every model to the benchmark table.
export_check = pd.read_csv(FORECAST_OUTPUT_PATH, parse_dates=["Month"])
pd.testing.assert_frame_equal(export_check, model_forecast_results)
for model_name, prediction_column in {
    "Random Forest": "Forecast_Demand",
    "Naive Forecast": "Naive_Forecast",
    "Seasonal Naive": "Seasonal_Naive_Forecast",
}.items():
    expected = benchmark_results.set_index("Model").loc[model_name]
    np.testing.assert_allclose(
        [
            mean_absolute_error(export_check["Actual_Demand"], export_check[prediction_column]),
            np.sqrt(mean_squared_error(export_check["Actual_Demand"], export_check[prediction_column])),
        ],
        expected[["MAE", "RMSE"]].to_numpy(dtype=float),
    )

print(f"Saved {len(model_forecast_results):,} rows: {FORECAST_OUTPUT_PATH}")
print("CSV round-trip and all three benchmark metrics verified.")
model_forecast_results.head()


Saved 192 rows: C:\Users\circl\Documents\New folder\Demand-Forecasting-Analytics\outputs\forecasts\model_forecast_results.csv
CSV round-trip and all three benchmark metrics verified.


,Month,Product_ID,Region_ID,Actual_Demand,Naive_Forecast,Seasonal_Naive_Forecast,Forecast_Demand,Forecast_Error,Absolute_Error,Squared_Error
0,2025-07-01,P001,R01,3906,6170.0,6193.0,6017.767134,2111.767134,2111.767134,4.459560e+06
1,2025-07-01,P001,R02,16164,13317.0,13842.0,11063.761102,-5100.238898,5100.238898,2.601244e+07
2,2025-07-01,P001,R03,1672,6697.0,11627.0,7401.136085,5729.136085,5729.136085,3.282300e+07
3,2025-07-01,P001,R04,13417,11271.0,8923.0,15253.319605,1836.319605,1836.319605,3.372070e+06
4,2025-07-01,P002,R01,10436,7962.0,8866.0,7285.875236,-3150.124764,3150.124764,9.923286e+06


# Stage 7: Error analysis by product, region and month

Analyze the saved historical test predictions without retraining or tuning.
MAE and RMSE are in demand units. WAPE (%) is `100 * sum(abs(error)) / sum(actual)`;
it is undefined for zero total demand. Bias is `mean(forecast - actual)`, so positive
means overforecast. Aggregate bias can hide offsetting errors.

Compare all three methods on identical observations within each segment.
For product-region series, flag a repeated direction when at least 5 of the 6
test months have errors of the same sign (not necessarily consecutive). This is
a descriptive review flag, not evidence of statistically established long-term bias.
Each series has only six observations. Do not use these test results for tuning;
validate any proposed model change or model-routing rule on development CV.


In [7]:
from IPython.display import display

# This section can also run independently using the exported CSV.
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "Python" / "run_sql_analysis.py").is_file()
)
error_data = pd.read_csv(
    PROJECT_ROOT / "outputs" / "forecasts" / "model_forecast_results.csv",
    parse_dates=["Month"],
)
assert not error_data.duplicated(["Month", "Product_ID", "Region_ID"]).any()
forecast_columns = {
    "Random Forest": "Forecast_Demand",
    "Naive Forecast": "Naive_Forecast",
    "Seasonal Naive": "Seasonal_Naive_Forecast",
}
assert np.isfinite(error_data[["Actual_Demand", *forecast_columns.values()]]).all().all()
np.testing.assert_allclose(
    error_data["Forecast_Error"],
    error_data["Forecast_Demand"] - error_data["Actual_Demand"],
)


def summarize_errors(data, keys):
    summaries = []
    for model_name, column in forecast_columns.items():
        work = data.assign(Error=data[column] - data["Actual_Demand"])
        work = work.assign(
            Abs_Error=work["Error"].abs(), Sq_Error=work["Error"] ** 2,
            Over=work["Error"].gt(0), Under=work["Error"].lt(0),
        )
        result = work.groupby(keys, as_index=False).agg(
            Observations=("Error", "size"),
            Actual_Total=("Actual_Demand", "sum"),
            Absolute_Error_Total=("Abs_Error", "sum"),
            MAE=("Abs_Error", "mean"), MSE=("Sq_Error", "mean"),
            Bias=("Error", "mean"),
            Overforecast_Count=("Over", "sum"),
            Underforecast_Count=("Under", "sum"),
        )
        result["RMSE"] = np.sqrt(result.pop("MSE"))
        result["WAPE_pct"] = 100 * result["Absolute_Error_Total"] / result["Actual_Total"].replace(0, np.nan)
        result["Model"] = model_name
        summaries.append(result)
    return pd.concat(summaries, ignore_index=True)


error_summaries = {
    "Product": summarize_errors(error_data, ["Product_ID"]),
    "Region": summarize_errors(error_data, ["Region_ID"]),
    "Month": summarize_errors(error_data, ["Month"]),
    "Product_Region": summarize_errors(error_data, ["Product_ID", "Region_ID"]),
}

# Every grouping must reconcile to the same overall absolute error per model.
for summary in error_summaries.values():
    for model_name, column in forecast_columns.items():
        rows = summary[summary["Model"].eq(model_name)]
        assert rows["Observations"].sum() == len(error_data)
        np.testing.assert_allclose(
            rows["Absolute_Error_Total"].sum(),
            (error_data[column] - error_data["Actual_Demand"]).abs().sum(),
        )

for dimension in ["Product", "Region", "Month"]:
    summary = error_summaries[dimension]
    rf = summary[summary["Model"].eq("Random Forest")].copy()
    rf = rf.sort_values("Month" if dimension == "Month" else "MAE", ascending=dimension == "Month")
    keys = {"Product": ["Product_ID"], "Region": ["Region_ID"], "Month": ["Month"]}[dimension]
    print(f"Random Forest errors by {dimension}")
    display(rf[keys + ["Observations", "MAE", "RMSE", "WAPE_pct", "Bias"]].round(2))

series_summary = error_summaries["Product_Region"]
series_errors = series_summary[series_summary["Model"].eq("Random Forest")].copy()
series_errors["Absolute_Error_Share_pct"] = (
    100 * series_errors["Absolute_Error_Total"] / series_errors["Absolute_Error_Total"].sum()
)
comparison = series_summary.pivot(
    index=["Product_ID", "Region_ID"], columns="Model", values="MAE"
).reset_index()
series_errors = series_errors.merge(comparison, on=["Product_ID", "Region_ID"], validate="one_to_one")
series_errors["Beats_Both_Baselines"] = (
    series_errors["MAE"].lt(series_errors["Naive Forecast"])
    & series_errors["MAE"].lt(series_errors["Seasonal Naive"])
)
series_errors = series_errors.sort_values("MAE", ascending=False).reset_index(drop=True)
print("Top 10 product-region series by MAE")
display(series_errors[[
    "Product_ID", "Region_ID", "MAE", "WAPE_pct", "Bias",
    "Absolute_Error_Share_pct", "Naive Forecast", "Seasonal Naive",
    "Beats_Both_Baselines",
]].head(10).round(2))

# Count direction by month, after verifying exactly one observation per series-month.
assert series_errors["Observations"].eq(6).all()
repeated_bias = series_errors[
    series_errors["Overforecast_Count"].ge(5) | series_errors["Underforecast_Count"].ge(5)
].copy()
print("Repeated error direction: at least 5 of 6 months")
display(repeated_bias[[
    "Product_ID", "Region_ID", "MAE", "WAPE_pct", "Bias",
    "Overforecast_Count", "Underforecast_Count",
]].round(2))

print("All series where Random Forest does not beat both baselines (MAE)")
display(series_errors.loc[~series_errors["Beats_Both_Baselines"], [
    "Product_ID", "Region_ID", "MAE", "Naive Forecast", "Seasonal Naive",
]].round(2))

print("Five largest individual forecast errors")
display(error_data.nlargest(5, "Absolute_Error")[[
    "Month", "Product_ID", "Region_ID", "Actual_Demand", "Forecast_Demand", "Forecast_Error",
]].round(2))

for baseline in ["Naive Forecast", "Seasonal Naive"]:
    wins = series_errors["MAE"].lt(series_errors[baseline]).sum()
    print(f"Random Forest beats {baseline}: {wins}/{len(series_errors)} series by MAE")
print(f"Overall WAPE: {100 * error_data['Absolute_Error'].sum() / error_data['Actual_Demand'].sum():.2f}%")
print(f"Overall mean bias: {error_data['Forecast_Error'].mean():.2f} units")


Random Forest errors by Product
Random Forest errors by Region
Random Forest errors by Month
Top 10 product-region series by MAE
Repeated error direction: at least 5 of 6 months
All series where Random Forest does not beat both baselines (MAE)
Five largest individual forecast errors
Random Forest beats Naive Forecast: 28/32 series by MAE
Random Forest beats Seasonal Naive: 22/32 series by MAE
Overall WAPE: 34.19%
Overall mean bias: -14.98 units


<ipython-input-1-96b4e4ed279d>:72: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(rf[keys + ["Observations", "MAE", "RMSE", "WAPE_pct", "Bias"]].round(2))
<ipython-input-1-96b4e4ed279d>:114: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].round(2))


,Product_ID,Observations,MAE,RMSE,WAPE_pct,Bias
3,P004,24,3937.52,5141.84,54.24,932.86
0,P001,24,3620.60,4704.38,31.35,-1022.27
2,P003,24,3618.57,5162.79,29.23,-1128.54
4,P005,24,3600.71,4829.67,41.33,-147.72
6,P007,24,3272.27,4077.30,38.04,-187.26
7,P008,24,2482.37,3020.28,28.70,41.74
1,P002,24,2304.92,3050.32,26.78,817.03
5,P006,24,2294.21,3256.34,29.62,574.33


,Region_ID,Observations,MAE,RMSE,WAPE_pct,Bias
3,R04,48,4509.23,5976.80,33.42,44.13
1,R02,48,3219.63,4069.19,31.99,-365.73
0,R01,48,2425.19,2996.72,32.66,-446.26
2,R03,48,2411.53,3292.63,41.80,707.94


,Month,Observations,MAE,RMSE,WAPE_pct,Bias
0,2025-07-01,32,3143.32,4435.04,36.09,-101.63
1,2025-08-01,32,3540.56,4367.14,40.30,-75.08
2,2025-09-01,32,3230.67,4114.30,35.69,161.77
3,2025-10-01,32,2670.90,3824.65,25.32,-1103.40
4,2025-11-01,32,2922.28,3959.44,34.55,1395.97
5,2025-12-01,32,3340.66,4709.49,34.88,-367.51


,Product_ID,Region_ID,MAE,WAPE_pct,Bias,Absolute_Error_Share_pct,Naive Forecast,Seasonal Naive,Beats_Both_Baselines
0,P004,R04,6955.75,63.57,36.05,6.92,8647.50,6496.83,False
1,P003,R04,6313.46,31.90,-4397.63,6.28,8944.67,8967.50,True
2,P005,R04,5507.00,45.97,1734.77,5.48,7743.83,4031.33,False
3,P005,R02,4625.07,44.95,-2581.71,4.60,5844.17,3481.17,False
4,P007,R04,4439.90,36.80,1809.61,4.42,3777.67,3089.33,False
5,P001,R04,4416.20,26.12,-3139.84,4.39,6330.00,3954.67,False
6,P001,R03,4409.36,56.12,-190.10,4.39,4958.50,4596.00,True
7,P003,R02,4119.27,30.94,-324.90,4.10,5995.17,3807.67,False
8,P006,R04,4069.05,42.21,3081.39,4.05,4938.17,6400.50,True
9,P007,R02,3468.56,40.61,-997.03,3.45,3692.33,4284.83,True


,Product_ID,Region_ID,MAE,WAPE_pct,Bias,Overforecast_Count,Underforecast_Count
8,P006,R04,4069.05,42.21,3081.39,5,1
12,P004,R03,3116.89,79.44,2747.44,5,1
14,P004,R02,2960.66,31.72,-713.30,1,5
26,P007,R01,1801.15,21.85,-1586.02,1,5
30,P005,R03,1282.45,26.39,955.65,5,1


,Product_ID,Region_ID,MAE,Naive Forecast,Seasonal Naive
0,P004,R04,6955.75,8647.50,6496.83
2,P005,R04,5507.00,7743.83,4031.33
3,P005,R02,4625.07,5844.17,3481.17
4,P007,R04,4439.90,3777.67,3089.33
5,P001,R04,4416.20,6330.00,3954.67
7,P003,R02,4119.27,5995.17,3807.67
12,P004,R03,3116.89,4085.50,1473.00
13,P005,R01,2988.32,2731.33,3943.00
15,P001,R02,2939.24,4101.33,1881.17
16,P003,R01,2769.29,4250.17,2610.83


,Month,Product_ID,Region_ID,Actual_Demand,Forecast_Demand,Forecast_Error
11,2025-07-01,P003,R04,31424,14682.86,-16741.14
179,2025-12-01,P005,R04,3264,15367.66,12103.66
163,2025-12-01,P001,R04,24643,12859.88,-11783.12
185,2025-12-01,P007,R02,17343,6621.68,-10721.32
107,2025-10-01,P003,R04,26405,16074.10,-10330.90


# Stage 8: Trace forecast errors back to source orders

Investigate P004 (Electronic Module) and P003 (Motor Component) in R04
(South America). Reconcile source orders to all saved monthly demand rows before
examining order counts, average order size, customer concentration, delivery
status and price. Keep delayed orders in demand, as in Stage 1.

Monthly demand = order count * mean order quantity. The change decomposition
uses symmetric weights: count contribution = change in count * mean of the two
months' average order sizes; size contribution = change in average size * mean
of the two months' order counts. They sum exactly to the demand change. This is
an accounting explanation, not causal attribution. Same-month order metrics
are diagnostic only and are not available as forecasting features in advance.


In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "Python" / "run_sql_analysis.py").is_file()
)
raw_data_dir = PROJECT_ROOT / "data" / "raw"
processed_data_dir = PROJECT_ROOT / "data" / "processed"
forecast_results_dir = PROJECT_ROOT / "outputs" / "forecasts"
source_orders = pd.read_csv(raw_data_dir / "sales_orders.csv", parse_dates=["Order_Date"])
source_orders["Month"] = source_orders["Order_Date"].dt.to_period("M").dt.to_timestamp()
assert source_orders["Order_ID"].is_unique
assert source_orders[["Order_Date", "Product_ID", "Region_ID", "Quantity"]].notna().all().all()
assert source_orders["Quantity"].gt(0).all()
keys = ["Month", "Product_ID", "Region_ID"]
saved_monthly = pd.read_csv(processed_data_dir / "monthly_demand.csv", parse_dates=["Month"])
source_monthly = source_orders.groupby(keys)["Quantity"].sum()
assert not saved_monthly.duplicated(keys).any()
assert source_monthly.index.isin(saved_monthly.set_index(keys).index).all()
np.testing.assert_allclose(
    source_monthly.reindex(saved_monthly.set_index(keys).index, fill_value=0),
    saved_monthly["Actual_Demand"],
)
print("Source order quantities reconcile to all saved monthly demand rows.")

focus_orders = source_orders[
    source_orders["Product_ID"].isin(["P003", "P004"]) & source_orders["Region_ID"].eq("R04")
].copy()
order_diagnostics = focus_orders.groupby(keys).agg(
    Actual_Demand=("Quantity", "sum"), Order_Count=("Order_ID", "size"),
    Average_Order_Quantity=("Quantity", "mean"), Largest_Order=("Quantity", "max"),
    Active_Customers=("Customer_ID", "nunique"), Revenue=("Revenue", "sum"),
).reset_index()
order_diagnostics["Weighted_Unit_Price"] = order_diagnostics["Revenue"] / order_diagnostics["Actual_Demand"]
order_diagnostics["Largest_Order_Share_pct"] = 100 * order_diagnostics["Largest_Order"] / order_diagnostics["Actual_Demand"]
customer_monthly = focus_orders.groupby(keys + ["Customer_ID"])["Quantity"].sum().rename("Customer_Quantity").reset_index()
customer_top = customer_monthly.groupby(keys)["Customer_Quantity"].max().rename("Largest_Customer_Quantity").reset_index()
order_diagnostics = order_diagnostics.merge(customer_top, on=keys, validate="one_to_one")
order_diagnostics["Largest_Customer_Share_pct"] = 100 * order_diagnostics["Largest_Customer_Quantity"] / order_diagnostics["Actual_Demand"]
order_diagnostics = order_diagnostics.sort_values(["Product_ID", "Region_ID", "Month"])
previous = order_diagnostics.groupby(["Product_ID", "Region_ID"])[
    ["Actual_Demand", "Order_Count", "Average_Order_Quantity"]
].shift(1)
order_diagnostics["Demand_Change"] = order_diagnostics["Actual_Demand"] - previous["Actual_Demand"]
order_diagnostics["Order_Count_Contribution"] = (
    (order_diagnostics["Order_Count"] - previous["Order_Count"])
    * (order_diagnostics["Average_Order_Quantity"] + previous["Average_Order_Quantity"]) / 2
)
order_diagnostics["Order_Size_Contribution"] = (
    (order_diagnostics["Average_Order_Quantity"] - previous["Average_Order_Quantity"])
    * (order_diagnostics["Order_Count"] + previous["Order_Count"]) / 2
)
valid = order_diagnostics["Demand_Change"].notna()
np.testing.assert_allclose(
    order_diagnostics.loc[valid, "Demand_Change"],
    order_diagnostics.loc[valid, "Order_Count_Contribution"] + order_diagnostics.loc[valid, "Order_Size_Contribution"],
    atol=1e-8,
)
predictions = pd.read_csv(forecast_results_dir / "model_forecast_results.csv", parse_dates=["Month"])
focus_test = order_diagnostics.merge(predictions, on=keys + ["Actual_Demand"], validate="one_to_one")
assert len(focus_test) == 12

def show_diagnostic_table(frame):
    shown = frame.copy()
    numeric = shown.select_dtypes(include="number").columns
    shown[numeric] = shown[numeric].round(2)
    display(shown)

for product in ["P004", "P003"]:
    print(f"{product} / R04: all 36 months of order history")
    show_diagnostic_table(order_diagnostics.loc[order_diagnostics["Product_ID"].eq(product), [
        "Month", "Actual_Demand", "Order_Count", "Average_Order_Quantity",
        "Active_Customers", "Largest_Order", "Largest_Order_Share_pct", "Largest_Customer_Share_pct",
    ]])
    print(f"{product} / R04: test predictions and monthly demand-change decomposition")
    show_diagnostic_table(focus_test.loc[focus_test["Product_ID"].eq(product), [
        "Month", "Actual_Demand", "Forecast_Demand", "Forecast_Error", "Order_Count",
        "Average_Order_Quantity", "Demand_Change", "Order_Count_Contribution", "Order_Size_Contribution",
    ]])

print("P003 / R04: customer contributions to the June-to-July demand increase")
customer_change = customer_monthly[
    customer_monthly["Product_ID"].eq("P003") & customer_monthly["Month"].isin(pd.to_datetime(["2025-06-01", "2025-07-01"]))
].pivot(index="Customer_ID", columns="Month", values="Customer_Quantity").fillna(0)
customer_change.columns = ["June_Quantity", "July_Quantity"]
customer_change["Quantity_Change"] = customer_change["July_Quantity"] - customer_change["June_Quantity"]
assert customer_change["Quantity_Change"].sum() == 9026
show_diagnostic_table(customer_change.sort_values("Quantity_Change", ascending=False).reset_index())

print("P003 / R04 / July: five largest source orders")
july_orders = focus_orders[focus_orders["Product_ID"].eq("P003") & focus_orders["Month"].eq("2025-07-01")]
show_diagnostic_table(july_orders.nlargest(5, "Quantity")[[
    "Order_ID", "Order_Date", "Customer_ID", "Quantity", "Unit_Price", "Order_Status",
]])
print("Test-month quantities by order status (delayed orders remain valid demand)")
show_diagnostic_table(focus_orders[focus_orders["Month"].ge("2025-07-01")].groupby(
    ["Product_ID", "Month", "Order_Status"]
)["Quantity"].sum().unstack(fill_value=0).reset_index())
print("Historical order-size reference: data before July 2025")
show_diagnostic_table(focus_orders[focus_orders["Month"].lt("2025-07-01")].groupby("Product_ID")["Quantity"].quantile([.5, .95, 1]).unstack().reset_index())
print("Weighted unit price by test month: descriptive only, no promotion indicator exists")
show_diagnostic_table(focus_test[["Product_ID", "Month", "Weighted_Unit_Price"]])


Source order quantities reconcile to all saved monthly demand rows.
P004 / R04: all 36 months of order history
P004 / R04: test predictions and monthly demand-change decomposition
P003 / R04: all 36 months of order history
P003 / R04: test predictions and monthly demand-change decomposition
P003 / R04: customer contributions to the June-to-July demand increase
P003 / R04 / July: five largest source orders
Test-month quantities by order status (delayed orders remain valid demand)
Historical order-size reference: data before July 2025
Weighted unit price by test month: descriptive only, no promotion indicator exists


,Month,Actual_Demand,Order_Count,Average_Order_Quantity,Active_Customers,Largest_Order,Largest_Order_Share_pct,Largest_Customer_Share_pct
1,2023-01-01,9213,9,1023.67,5,1816,19.71,36.39
3,2023-02-01,16053,13,1234.85,10,1971,12.28,27.94
5,2023-03-01,17281,19,909.53,12,1894,10.96,17.05
7,2023-04-01,7893,11,717.55,10,1649,20.89,20.89
9,2023-05-01,16257,17,956.29,11,1818,11.18,19.17
11,2023-06-01,10611,13,816.23,10,1998,18.83,31.28
13,2023-07-01,15872,15,1058.13,11,1888,11.90,19.01
15,2023-08-01,7167,7,1023.86,6,1566,21.85,25.69
17,2023-09-01,3970,6,661.67,5,1215,30.60,39.07
19,2023-10-01,17217,17,1012.76,11,1920,11.15,20.72


,Month,Actual_Demand,Forecast_Demand,Forecast_Error,Order_Count,Average_Order_Quantity,Demand_Change,Order_Count_Contribution,Order_Size_Contribution
6,2025-07-01,6207,9780.69,3573.69,5,1241.40,-1482.0,-5025.75,3543.75
7,2025-08-01,13092,10434.08,-2657.92,9,1454.67,6885.0,5392.13,1492.87
8,2025-09-01,5721,13187.41,7466.41,7,817.29,-7371.0,-2271.95,-5099.05
9,2025-10-01,16810,8819.47,-7990.53,16,1050.62,11089.0,8405.60,2683.40
10,2025-11-01,5192,15127.29,9935.29,6,865.33,-11618.0,-9579.79,-2038.21
11,2025-12-01,18632,8521.32,-10110.68,16,1164.50,13440.0,10149.17,3290.83


,Month,Actual_Demand,Order_Count,Average_Order_Quantity,Active_Customers,Largest_Order,Largest_Order_Share_pct,Largest_Customer_Share_pct
0,2023-01-01,16977,14,1212.64,11,1971,11.61,18.90
2,2023-02-01,12112,12,1009.33,7,1970,16.26,31.18
4,2023-03-01,12210,15,814.00,10,1944,15.92,32.04
6,2023-04-01,16666,15,1111.07,10,1955,11.73,28.05
8,2023-05-01,10844,10,1084.40,8,1706,15.73,22.66
10,2023-06-01,9007,8,1125.88,8,1660,18.43,18.43
12,2023-07-01,11687,12,973.92,9,1959,16.76,20.52
14,2023-08-01,12649,11,1149.91,8,1831,14.48,18.97
16,2023-09-01,11995,13,922.69,9,1416,11.80,28.36
18,2023-10-01,18189,19,957.32,13,1777,9.77,21.53


,Month,Actual_Demand,Forecast_Demand,Forecast_Error,Order_Count,Average_Order_Quantity,Demand_Change,Order_Count_Contribution,Order_Size_Contribution
0,2025-07-01,31424,14682.86,-16741.14,27,1163.85,9026.0,7993.13,1032.87
1,2025-08-01,13825,15036.88,1211.88,14,987.50,-17599.0,-13983.79,-3615.21
2,2025-09-01,17786,13932.28,-3853.72,15,1185.73,3961.0,1086.62,2874.38
3,2025-10-01,26405,16074.10,-10330.90,24,1100.21,8619.0,10286.74,-1667.74
4,2025-11-01,17350,16142.51,-1207.49,19,913.16,-9055.0,-5033.42,-4021.58
5,2025-12-01,11942,16477.59,4535.59,13,918.62,-5408.0,-5495.32,87.32


,Customer_ID,June_Quantity,July_Quantity,Quantity_Change
0,C028,0.0,5742.0,5742.0
1,C026,1269.0,4171.0,2902.0
2,C050,1089.0,3709.0,2620.0
3,C037,220.0,2494.0,2274.0
4,C006,0.0,1919.0,1919.0
5,C019,1335.0,2731.0,1396.0
6,C002,0.0,1369.0,1369.0
7,C005,0.0,1349.0,1349.0
8,C023,198.0,1513.0,1315.0
9,C047,1951.0,1945.0,-6.0


,Order_ID,Order_Date,Customer_ID,Quantity,Unit_Price,Order_Status
2588,SO02589,2025-07-02,C047,1945,95.01,Completed
1422,SO01423,2025-07-13,C006,1919,430.44,Completed
3819,SO03820,2025-07-03,C050,1863,89.47,Completed
5904,SO05905,2025-07-31,C050,1846,496.15,Completed
5890,SO05891,2025-07-25,C026,1713,442.14,Completed


Order_Status,Product_ID,Month,Completed,Delayed
0,P003,2025-07-01,31424,0
1,P003,2025-08-01,12157,1668
2,P003,2025-09-01,12418,5368
3,P003,2025-10-01,26198,207
4,P003,2025-11-01,16589,761
5,P003,2025-12-01,10478,1464
6,P004,2025-07-01,4622,1585
7,P004,2025-08-01,11095,1997
8,P004,2025-09-01,5721,0
9,P004,2025-10-01,12537,4273


,Product_ID,0.5,0.95,1.0
0,P003,1135.0,1933.50,1988.0
1,P004,899.0,1882.05,1998.0


,Product_ID,Month,Weighted_Unit_Price
0,P003,2025-07-01,311.77
1,P003,2025-08-01,289.16
2,P003,2025-09-01,308.09
3,P003,2025-10-01,299.17
4,P003,2025-11-01,254.24
5,P003,2025-12-01,176.98
6,P004,2025-07-01,274.49
7,P004,2025-08-01,213.08
8,P004,2025-09-01,406.04
9,P004,2025-10-01,272.90


## Findings and follow-up

- **P003 / R04, July:** demand increased from 22,398 to 31,424 units (+40.3%).
  Orders rose from 20 to 27 (+35%); average quantity rose from 1,119.90 to
  1,163.85 (+3.9%). The symmetric decomposition attributes about 88.6% of the
  month-over-month increase to order count and 11.4% to average order size.
  The largest order was 1,945 units (6.2% of demand), below the pre-test maximum
  of 1,988. Demand was distributed across 14 customers; the largest contributed
  18.3%. All July orders were Completed. This is a higher-order-volume month,
  without evidence of a single exceptional order driving the peak.
- **P003 / R04, October:** 24 orders versus 15 in September, while average order
  quantity decreased from 1,185.73 to 1,100.21. The second missed peak also
  coincides with a substantial increase in order count.
- **P004 / R04:** July-December order counts were 5, 9, 7, 16, 6 and 16; demand
  was 6,207, 13,092, 5,721, 16,810, 5,192 and 18,632 units. Predictions were
  high in low-demand months and low in high-demand months. All five consecutive
  test-month demand changes had the opposite sign to the prediction changes.
  The largest test order (1,997 units) was below the pre-test maximum (1,998).
  December demand slightly exceeded the prior monthly maximum (18,008), but
  neither its order count (16) nor average size (1,164.50) exceeded prior maxima.
  September was concentrated: two customers supplied about 84.0% of demand.
- **Limits:** the data contains prices and delivery status, but no promotion,
  stockout, campaign or advance-order snapshot fields. Those business causes
  cannot be established. Historical and current order-volume patterns describe
  observed demand; actual same-month counts must not enter advance forecasts.
- **Model review:** the current feature list uses historical demand and calendar
  fields but omits Product_ID and Region_ID. Check whether adding encoded series
  identifiers and lagged order count/size/customer features improves development
  cross-validation. This is a hypothesis, not a confirmed explanation of the
  errors. Do not delete these valid orders as outliers or tune on the test set.
